In [6]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# End-to-End Machine Learning Project: Titanic Survival Prediction\n",
    "**Dataset Chosen:** Titanic (Binary Classification)\n",
    "\n",
    "This notebook follows the end-to-end ML workflow to predict passenger survival on the Titanic. We will perform Exploratory Data Analysis (EDA), preprocess the data, and compare three different machine learning models using cross-validation.\n",
    "\n",
    "## 1. Problem Definition & Dataset Selection\n",
    "* **Business Objective:** Understand the factors that influenced survival on the Titanic and build a model to predict whether a passenger survived or not.\n",
    "* **ML Framing:** Supervised Learning -> Binary Classification ($Y \\in \\{0, 1\\}$).\n",
    "* **Success Metric:** Accuracy and F1-Score (to balance Precision and Recall)."
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold\n",
    "from sklearn.preprocessing import StandardScaler, OneHotEncoder\n",
    "from sklearn.compose import ColumnTransformer\n",
    "from sklearn.pipeline import Pipeline\n",
    "from sklearn.impute import SimpleImputer\n",
    "\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.svm import SVC\n",
    "from sklearn.metrics import accuracy_score, classification_report, confusion_matrix\n",
    "\n",
    "# Suppress warnings for cleaner output\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Load the Titanic dataset directly from seaborn\n",
    "df = sns.load_dataset('titanic')\n",
    "\n",
    "# Display basic information\n",
    "print(df.info())\n",
    "print(\"\\nFirst 5 rows:\")\n",
    "display(df.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Exploratory Data Analysis (EDA)\n",
    "We need to understand the distributions of our features and their relationship with the target variable (`survived`)."
   ]
  },
  {
   "cell_type": "code",
  
   "metadata": {},
   "outputs": [],
   "source": [
    "# Set up the matplotlib figure layout\n",
    "fig, axes = plt.subplots(3, 2, figsize=(15, 18))\n",
    "plt.subplots_adjust(hspace=0.4)\n",
    "\n",
    "# 1. Target Distribution (Univariate)\n",
    "sns.countplot(data=df, x='survived', palette='Set2', ax=axes[0, 0])\n",
    "axes[0, 0].set_title('1. Distribution of Survival (Target Variable)')\n",
    "axes[0, 0].set_xticklabels(['Did Not Survive (0)', 'Survived (1)'])\n",
    "\n",
    "# 2. Survival by Sex (Bivariate)\n",
    "sns.countplot(data=df, x='survived', hue='sex', palette='Pastel1', ax=axes[0, 1])\n",
    "axes[0, 1].set_title('2. Survival by Passenger Sex')\n",
    "\n",
    "# 3. Survival by Passenger Class (Bivariate)\n",
    "sns.countplot(data=df, x='pclass', hue='survived', palette='Set1', ax=axes[1, 0])\n",
    "axes[1, 0].set_title('3. Survival by Passenger Class (Pclass)')\n",
    "\n",
    "# 4. Age Distribution by Survival (Multivariate/Distribution)\n",
    "sns.histplot(data=df, x='age', hue='survived', kde=True, palette='coolwarm', ax=axes[1, 1])\n",
    "axes[1, 1].set_title('4. Age Distribution grouped by Survival')\n",
    "\n",
    "# 5. Correlation Heatmap (Multivariate)\n",
    "# Select only numerical columns for correlation\n",
    "numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns\n",
    "corr_matrix = df[numerical_cols].corr()\n",
    "\n",
    "sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', vmin=-1, vmax=1, ax=axes[2, 0])\n",
    "axes[2, 0].set_title('5. Correlation Heatmap of Numerical Features')\n",
    "\n",
    "# Hide the empty subplot (bottom right)\n",
    "fig.delaxes(axes[2,1])\n",
    "\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### EDA Findings:\n",
    "1. **Target Distribution:** The dataset is slightly imbalanced; more people died than survived.\n",
    "2. **Sex:** Females had a significantly higher survival rate than males.\n",
    "3. **Pclass:** 1st-class passengers had the highest survival rate, while 3rd-class passengers had the lowest.\n",
    "4. **Age:** Children (age < 10) had a higher chance of survival. A large peak of non-survivors is seen around the age of 20-30.\n",
    "5. **Correlation:** `pclass` and `fare` have a moderate negative correlation with survival, aligning with the class visualization.\n",
    "\n",
    "## 3. Data Preprocessing & Feature Engineering\n",
    "We will drop redundant columns, handle missing values (Age, Embarked), and encode categorical variables (Sex, Embarked)."
   ]
  },
  {
   "cell_type": "code",
   
   "metadata": {},
   "outputs": [],
   "source": [
    "# Drop redundant or highly missing columns\n",
    "cols_to_drop = ['deck', 'alive', 'class', 'who', 'adult_male', 'embark_town', 'parch', 'sibsp']\n",
    "# Note: we drop 'alive' because it's exactly the same as 'survived'\n",
    "df_clean = df.drop(columns=cols_to_drop)\n",
    "\n",
    "# Define feature matrix (X) and target vector (y)\n",
    "X = df_clean.drop('survived', axis=1)\n",
    "y = df_clean['survived']\n",
    "\n",
    "# Define categorical and numerical features\n",
    "categorical_features = ['sex', 'embarked']\n",
    "numerical_features = ['pclass', 'age', 'fare']\n",
    "\n",
    "# Create preprocessing pipelines\n",
    "numeric_transformer = Pipeline(steps=[\n",
    "    ('imputer', SimpleImputer(strategy='median')),\n",
    "    ('scaler', StandardScaler())\n",
    "])\n",
    "\n",
    "categorical_transformer = Pipeline(steps=[\n",
    "    ('imputer', SimpleImputer(strategy='most_frequent')),\n",
    "    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))\n",
    "])\n",
    "\n",
    "# Combine into a ColumnTransformer\n",
    "preprocessor = ColumnTransformer(\n",
    "    transformers=[\n",
    "        ('num', numeric_transformer, numerical_features),\n",
    "        ('cat', categorical_transformer, categorical_features)\n",
    "    ])\n",
    "\n",
    "print(\"Preprocessing pipeline established successfully.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Modelling and Cross-Validation\n",
    "We will build three models: Logistic Regression (Baseline), Random Forest, and a Support Vector Classifier (SVC). We will evaluate them using 5-Fold Cross-Validation."
   ]
  },
  {
   "cell_type": "code",
  
   "metadata": {},
   "outputs": [],
   "source": [
    "# Define models\n",
    "models = {\n",
    "    \"Logistic Regression\": LogisticRegression(random_state=42),\n",
    "    \"Random Forest\": RandomForestClassifier(n_estimators=100, random_state=42),\n",
    "    \"Support Vector Machine\": SVC(random_state=42)\n",
    "}\n",
    "\n",
    "# Set up Cross-Validation\n",
    "cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)\n",
    "\n",
    "# Store results\n",
    "results = []\n",
    "\n",
    "print(\"--- Model Cross-Validation Results (Accuracy) ---\")\n",
    "for name, model in models.items():\n",
    "    # Create a pipeline that combines preprocessor and model\n",
    "    clf = Pipeline(steps=[('preprocessor', preprocessor),\n",
    "                          ('classifier', model)])\n",
    "    \n",
    "    # Perform cross-validation\n",
    "    cv_scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')\n",
    "    \n",
    "    results.append({\n",
    "        'Model': name,\n",
    "        'Mean Accuracy': cv_scores.mean(),\n",
    "        'Std Dev': cv_scores.std()\n",
    "    })\n",
    "    \n",
    "    print(f\"{name}: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})\")\n",
    "\n",
    "# Convert results to DataFrame for nice formatting\n",
    "results_df = pd.DataFrame(results).sort_values(by='Mean Accuracy', ascending=False)\n",
    "display(results_df)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### Final Model Evaluation on a Hold-Out Test Set"
   ]
  },
  {
   "cell_type": "code",
  
   "metadata": {},
   "outputs": [],
   "source": [
    "# Split data into train and test sets (80/20)\n",
    "X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\n",
    "\n",
    "# We choose the Support Vector Machine (or RF) based on CV results\n",
    "best_model = Pipeline(steps=[('preprocessor', preprocessor),\n",
    "                             ('classifier', SVC(random_state=42))])\n",
    "\n",
    "# Train the best model on the full training set\n",
    "best_model.fit(X_train, y_train)\n",
    "\n",
    "# Predict on the test set\n",
    "y_pred = best_model.predict(X_test)\n",
    "\n",
    "# Report results\n",
    "print(\"--- Final Model Evaluation on Test Set ---\")\n",
    "print(f\"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}\\n\")\n",
    "\n",
    "print(\"Classification Report:\")\n",
    "print(classification_report(y_test, y_pred))\n",
    "\n",
    "# Plot Confusion Matrix\n",
    "cm = confusion_matrix(y_test, y_pred)\n",
    "plt.figure(figsize=(6,4))\n",
    "sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Died', 'Survived'], yticklabels=['Died', 'Survived'])\n",
    "plt.ylabel('Actual')\n",
    "plt.xlabel('Predicted')\n",
    "plt.title('Confusion Matrix: Support Vector Machine')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Conclusion\n",
    "\n",
    "* **Insights:** Sex and Passenger Class were the most significant indicators of survival.\n",
    "* **Model Performance:** The Support Vector Machine and Random Forest classifiers performed best during cross-validation, achieving roughly ~81-82% accuracy.\n",
    "* **Next Steps:** To improve the model further, we could implement hyperparameter tuning (using `GridSearchCV`), extract titles from passenger names (e.g., \"Mr.\", \"Mrs.\", \"Dr.\") as a new feature, or experiment with ensemble methods like XGBoost."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}

{'cells': [{'cell_type': 'markdown',
   'metadata': {},
   'source': ['# End-to-End Machine Learning Project: Titanic Survival Prediction\n',
    '**Dataset Chosen:** Titanic (Binary Classification)\n',
    '\n',
    'This notebook follows the end-to-end ML workflow to predict passenger survival on the Titanic. We will perform Exploratory Data Analysis (EDA), preprocess the data, and compare three different machine learning models using cross-validation.\n',
    '\n',
    '## 1. Problem Definition & Dataset Selection\n',
    '* **Business Objective:** Understand the factors that influenced survival on the Titanic and build a model to predict whether a passenger survived or not.\n',
    '* **ML Framing:** Supervised Learning -> Binary Classification ($Y \\in \\{0, 1\\}$).\n',
    '* **Success Metric:** Accuracy and F1-Score (to balance Precision and Recall).']},
  {'cell_type': 'code',
   'metadata': {},
   'outputs': [],
   'source': ['import pandas as pd\n',
    'import numpy as np\n'